# Cross-Modal Alignment in Image Captioning
**CS444 Final Project**

## 0. Setup

In [ ]:
!pip install -r requirements.txt
import sys
sys.path.insert(0, 'src')

import torch
import pandas as pd
from pathlib import Path

from image_captioning.config import Config
from image_captioning.data import build_dataloaders, dataset_summary, load_coco_annotations, sample_subset
from image_captioning.modeling import build_model, load_processors, parameter_count
from image_captioning.training import run_training
from image_captioning.generation import generate_captions
from image_captioning.evaluation import evaluate
from image_captioning.visualization import (
    plot_loss_curve, plot_metric_bars, plot_results_table,
    show_sample_images, show_predictions_comparison, show_model_summary
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Hyperparameters

In [ ]:
# ── All tunable parameters are here ──────────────────────────────────────────
TRAIN_SUBSET_SIZE = 30_000
VAL_SIZE          = 5_000
BATCH_SIZE        = 32
NUM_EPOCHS        = 5
LEARNING_RATE     = 5e-5
MAX_LENGTH        = 40
NUM_BEAMS         = 4
SEED              = 42
DATA_ROOT         = Path('/workspace/data/coco')
OUTPUT_ROOT       = Path('/workspace/outputs')
# ─────────────────────────────────────────────────────────────────────────────

cfg = Config(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    train_subset_size=TRAIN_SUBSET_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    max_length=MAX_LENGTH,
    num_beams=NUM_BEAMS,
    seed=SEED,
)

## 2. Data

In [ ]:
# Load annotations and build reference map
train_rows_all = load_coco_annotations(cfg.train_ann_file)
val_rows_all   = load_coco_annotations(cfg.val_ann_file)
train_rows = sample_subset(train_rows_all, cfg.train_subset_size, cfg.seed)
val_rows   = sample_subset(val_rows_all,   cfg.val_size,          cfg.seed)

display(dataset_summary(train_rows, val_rows))

In [ ]:
# Sample images preview
show_sample_images(val_rows, cfg.val_image_dir, n=4)

## 3. Models

In [ ]:
# Parameter count summary for all 4 model configurations
model_infos = []
for exp in cfg.experiments:
    ip, tok = load_processors(exp['encoder'])
    m = build_model(exp['encoder'], exp['use_mapper'], tok, DEVICE)
    total, trainable = parameter_count(m)
    model_infos.append([
        exp['name'],
        exp['encoder'].split('/')[-1],
        'MLP' if exp['use_mapper'] else 'None',
        f'{total:,}',
        f'{trainable:,}',
    ])
    del m
    torch.cuda.empty_cache()

display(show_model_summary(model_infos))

## 4. Experiments

### 4.1 Vanilla ViT + No Mapper

In [ ]:
exp1 = cfg.experiments[0]
image_processor_1, tokenizer_1 = load_processors(exp1['encoder'])
train_loader_1, val_loader_1, _, val_ref_map = build_dataloaders(cfg, image_processor_1, tokenizer_1)
model_1 = build_model(exp1['encoder'], exp1['use_mapper'], tokenizer_1, DEVICE)

history_1 = run_training(model_1, train_loader_1, val_loader_1, cfg, exp1['name'])
plot_loss_curve(history_1, exp1['name'])

In [ ]:
preds_1 = generate_captions(model_1, val_rows, cfg.val_image_dir, image_processor_1, tokenizer_1, cfg, DEVICE)
show_sample_images(val_rows[:4], cfg.val_image_dir)

### 4.2 Vanilla ViT + MLP Mapper

In [ ]:
exp2 = cfg.experiments[1]
image_processor_2, tokenizer_2 = load_processors(exp2['encoder'])
train_loader_2, val_loader_2, _, _ = build_dataloaders(cfg, image_processor_2, tokenizer_2)
model_2 = build_model(exp2['encoder'], exp2['use_mapper'], tokenizer_2, DEVICE)

history_2 = run_training(model_2, train_loader_2, val_loader_2, cfg, exp2['name'])
plot_loss_curve(history_2, exp2['name'])

In [ ]:
preds_2 = generate_captions(model_2, val_rows, cfg.val_image_dir, image_processor_2, tokenizer_2, cfg, DEVICE)
show_sample_images(val_rows[:4], cfg.val_image_dir)

### 4.3 CLIP ViT + No Mapper

In [ ]:
exp3 = cfg.experiments[2]
image_processor_3, tokenizer_3 = load_processors(exp3['encoder'])
train_loader_3, val_loader_3, _, _ = build_dataloaders(cfg, image_processor_3, tokenizer_3)
model_3 = build_model(exp3['encoder'], exp3['use_mapper'], tokenizer_3, DEVICE)

history_3 = run_training(model_3, train_loader_3, val_loader_3, cfg, exp3['name'])
plot_loss_curve(history_3, exp3['name'])

In [ ]:
preds_3 = generate_captions(model_3, val_rows, cfg.val_image_dir, image_processor_3, tokenizer_3, cfg, DEVICE)
show_sample_images(val_rows[:4], cfg.val_image_dir)

### 4.4 CLIP ViT + MLP Mapper

In [ ]:
exp4 = cfg.experiments[3]
image_processor_4, tokenizer_4 = load_processors(exp4['encoder'])
train_loader_4, val_loader_4, _, _ = build_dataloaders(cfg, image_processor_4, tokenizer_4)
model_4 = build_model(exp4['encoder'], exp4['use_mapper'], tokenizer_4, DEVICE)

history_4 = run_training(model_4, train_loader_4, val_loader_4, cfg, exp4['name'])
plot_loss_curve(history_4, exp4['name'])

In [ ]:
preds_4 = generate_captions(model_4, val_rows, cfg.val_image_dir, image_processor_4, tokenizer_4, cfg, DEVICE)
show_sample_images(val_rows[:4], cfg.val_image_dir)

## 5. Evaluate

In [ ]:
all_results = []
for preds, exp in zip([preds_1, preds_2, preds_3, preds_4], cfg.experiments):
    scores = evaluate(preds, val_ref_map)
    all_results.append({'Experiment': exp['name'], **scores})

results_df = pd.DataFrame(all_results)
display(plot_results_table(results_df))

In [ ]:
plot_metric_bars(results_df)

## 6. Qualitative Analysis

In [ ]:
# Side-by-side comparison: same images, all 4 models
sample_ids = list(preds_1.keys())[:4]
predictions_dict = {
    'vit_no_mapper':   preds_1,
    'vit_mlp_mapper':  preds_2,
    'clip_no_mapper':  preds_3,
    'clip_mlp_mapper': preds_4,
}
show_predictions_comparison(sample_ids, predictions_dict, val_ref_map, cfg.val_image_dir, val_rows, n=4)

## 7. Discussion

*(Write analysis here after experiments are complete.)*

**Key questions to address:**
1. Does CLIP pre-alignment improve over Vanilla ViT?
2. Does the MLP Mapper add value on top of the encoder alone?
3. Does CLIP + MLP Mapper combine well, or does one dominate?
4. What do failure cases reveal about cross-modal alignment?